Test panel composition, conformity and criterion recovery mechanisms.....


In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import io
import json
import os
import re
import sys
import warnings
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings('ignore')
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/panel_mechanisms/criterion_recovery_and_difficulty'))
os.makedirs(OUT, exist_ok=True)
PANEL_MEMBERS = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog_gpt': ['gpt'], 'homog_ds': ['deepseekv4']}
ARM_PATH = {'P-base': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/delphi_pweak_noroles_judge.csv', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/delphi_pmixed_noroles_judge.csv', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/delphi_pstrong_noroles_judge.csv', 'homog_gpt': 'experiments/homogeneous_panels/three_gpt_members/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog_ds': 'experiments/homogeneous_panels/three_deepseekv4_members_complete/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv', 'USC@7': 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc7_judge.csv'}
SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
for m in SINGLES:
    ARM_PATH['base_' + m] = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_noroles_%s_judge.csv' % (m, m)
ROUNDS = {'P-base': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete/llm_as_judge/round_evaluation/delphi_base_noroles_protocol_r%d_judge.csv', 'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/round_evaluation/delphi_pweak_noroles_protocol_r%d_judge.csv', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/round_evaluation/delphi_pmixed_noroles_protocol_r%d_judge.csv', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/round_evaluation/delphi_pstrong_noroles_protocol_r%d_judge.csv', 'homog_gpt': 'experiments/homogeneous_panels/three_gpt_members/llm_as_judge/round_evaluation/delphi_homog_gpt_noroles_protocol_r%d_judge.csv', 'homog_ds': 'experiments/homogeneous_panels/three_deepseekv4_members_complete/llm_as_judge/round_evaluation/delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv'}

def rd(rel):
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        return None
    d = read_csv_normalized(p, sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    if 'Judge_score' not in d.columns:
        return None
    return d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette').set_index('Vignette')

def item_frame(d):
    r = {}
    for v, s in d['judge_details'].dropna().items():
        try:
            arr = json.loads(s)
        except Exception:
            continue
        for it in arr:
            ax = next((t.split(':', 1)[1] for t in it.get('tags', []) if isinstance(t, str) and t.startswith('axis:')), None)
            r[v, it['id']] = (bool(it.get('met')), it.get('points', 0) or 0, ax)
    return r

def criterion_texts(d):
    out = {}
    for v, blob in d['Criterion'].dropna().items():
        parts = re.split('(?m)^\\[(\\d+)\\]\\n', str(blob).replace('\r\n', '\n'))
        for k in range(1, len(parts), 2):
            m = re.match('points:\\s*\\S+\\ntags:[^\\n]*\\ncriterion:\\n(.*)$', parts[k + 1], re.S)
            if m:
                out[v, int(parts[k])] = m.group(1).strip()
    return out
print('=' * 108)
print('ЗАГРУЗКА ПОПУНКТНЫХ ВЕРДИКТОВ')
print('=' * 108)
RAW, ITEMS = ({}, {})
for a, p in ARM_PATH.items():
    d = rd(p)
    if d is None or 'judge_details' not in d.columns:
        continue
    RAW[a] = d
    ITEMS[a] = item_frame(d)
print('плеч загружено: %d  (%s)' % (len(ITEMS), ', '.join(ITEMS)))
TEXTS = criterion_texts(RAW['P-base'])
common = set(ITEMS['P-base'])
for a in ITEMS:
    common &= set(ITEMS[a])
POS = sorted([k for k in common if ITEMS['P-base'][k][1] > 0])
print('сопоставлено положительных пунктов во ВСЕХ %d плечах: %d' % (len(ITEMS), len(POS)))

def h32():
    mem = PANEL_MEMBERS['P-base']
    outside = [m for m in SINGLES if m not in mem]
    rows = []
    for k in POS:
        km = sum((ITEMS['base_' + m][k][0] for m in mem))
        ext = np.mean([ITEMS['base_' + m][k][0] for m in outside])
        rows.append({'case': k[0], 'cid': k[1], 'k_mem': km, 'panel': ITEMS['P-base'][k][0], 'points': ITEMS['P-base'][k][1], 'axis': ITEMS['P-base'][k][2], 'ext_pass': ext, 'text': TEXTS.get(k, '')})
    d = pd.DataFrame(rows)
    z = d[d.k_mem == 0].copy()
    z['recovered'] = z['panel'].astype(int)
    print('\n' + '=' * 108)
    print('H3.2  ЧТО ИМЕННО ПОРОЖДАЕТ ПАНЕЛЬ')
    print('=' * 108)
    print('Подвыборка: пункты, которые НЕ выполнил ни один из трёх участников панели. n = %d, восстановлено %d (%.1f%%).' % (len(z), z.recovered.sum(), 100 * z.recovered.mean()))
    print('Внешняя трудность ext_pass -- доля из 4 плеч ВНЕ панели (deepseekv4pro, gemini, glm, kimi), выполнивших пункт. Циркулярности нет: эти плечи в панель не входят.')
    print('\n[1] ВОССТАНОВЛЕНИЕ ПО ВНЕШНЕЙ ТРУДНОСТИ')
    z['ext_bin'] = pd.cut(z.ext_pass, [-0.01, 0.001, 0.26, 0.51, 0.76, 1.01], labels=['0/4 никто вне панели', '1/4', '2/4', '3/4', '4/4 все вне панели'])
    t1 = z.groupby('ext_bin', observed=True).agg(пунктов=('recovered', 'size'), восстановлено=('recovered', 'mean'), средний_вес=('points', 'mean'))
    t1['восстановлено_шт'] = z.groupby('ext_bin', observed=True)['recovered'].sum()
    print(t1.to_string())
    ct = pd.crosstab(z.ext_bin, z.recovered)
    chi = stats.chi2_contingency(ct)
    rho = stats.spearmanr(z.ext_pass, z.recovered)
    print('  chi2 по стратам: p = %.3g | Spearman(ext_pass, recovered) = %+.3f, p = %.3g' % (chi.pvalue, rho.statistic, rho.pvalue))
    hard = z[z.ext_pass == 0]
    lo, hi = stats.beta.ppf([0.025, 0.975], [hard.recovered.sum(), hard.recovered.sum() + 1], [len(hard) - hard.recovered.sum() + 1, len(hard) - hard.recovered.sum()])
    print('  КЛЮЧЕВОЕ: среди пунктов, которые не выполнило НИ ОДНО из 7 одиночных плеч,')
    print('  панель выполняет %d/%d = %.3f, 95%% ДИ [%.3f; %.3f]' % (hard.recovered.sum(), len(hard), hard.recovered.mean(), lo, hi))
    print('\n[2] ВОССТАНОВЛЕНИЕ ПО ВЕСУ ПУНКТА (points)')
    z['w_bin'] = pd.cut(z.points, [0, 3, 5, 7, 20], labels=['1-3', '4-5', '6-7', '8+'])
    print(z.groupby('w_bin', observed=True).agg(пунктов=('recovered', 'size'), восстановлено=('recovered', 'mean')).to_string())
    r = stats.spearmanr(z.points, z.recovered)
    print('  Spearman(points, recovered) = %+.3f, p = %.3g' % (r.statistic, r.pvalue))
    print('\n[3] СОВМЕСТНАЯ ЛОГИСТИЧЕСКАЯ МОДЕЛЬ (что предсказывает восстановление)')
    z['case_lvl'] = z.groupby('case')['ext_pass'].transform('mean')
    m = smf.logit('recovered ~ ext_pass + points + case_lvl + C(axis)', z).fit(disp=0)
    tab = pd.DataFrame({'коэф': m.params, 'SE': m.bse, 'z': m.tvalues, 'p': m.pvalues, 'OR': np.exp(m.params)})
    print(tab.to_string())
    print('  псевдо-R2 = %.3f, n = %d' % (m.prsquared, int(m.nobs)))
    print('\n[4] ПО ОСЯМ РУБРИКИ')
    t4 = z.groupby('axis').agg(пунктов=('recovered', 'size'), восстановлено=('recovered', 'mean'), внешняя_трудность=('ext_pass', 'mean'))
    print(t4.to_string())
    print('\n[5] ЛЕКСИКА: чем восстановленные пункты отличаются от невосстановленных')
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        vz = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), min_df=4)
        X = vz.fit_transform(z.text.fillna(''))
        y = z.recovered.values
        voc = np.array(vz.get_feature_names_out())
        d1 = np.asarray(X[y == 1].mean(axis=0)).ravel()
        d0 = np.asarray(X[y == 0].mean(axis=0)).ravel()
        diff = d1 - d0
        o = np.argsort(diff)
        print('  чаще у ВОССТАНОВЛЕННЫХ: ' + ', '.join(voc[o[-12:]][::-1]))
        print('  чаще у ПРОПУЩЕННЫХ:     ' + ', '.join(voc[o[:12]]))
    except Exception as e:
        print('  (пропущено: %s)' % e)
    z.drop(columns=['text']).to_csv(os.path.normpath(os.path.join(OUT, 'criteria_recovered_from_zero_successful_members.csv')), index=False, encoding='utf-8-sig')
    return z

def h33():
    print('\n' + '=' * 108)
    print('H3.3  ЦЕНА ОСТАНОВКИ ПОСЛЕ R1 КАК ПОЛИТИКА')
    print('=' * 108)
    print('Политика «как есть» = ранняя остановка по единогласию (то, что прогонялось).')
    print('Токены политики «стоп@Rk» = все вызовы экспертов по раунды 1..k + сводки')
    print('организатора после этих раундов + один финальный синтез.')
    rows = []
    for arm, pat in ROUNDS.items():
        fin = RAW.get(arm)
        r1, r2 = (rd(pat % 1), rd(pat % 2))
        if fin is None or r1 is None:
            continue
        ix = fin.index.intersection(r1.index)
        if r2 is not None:
            ix = ix.intersection(r2.index)
        f = fin.loc[ix]

        def tk(pref):
            c = [x for x in f.columns if re.match('%s_total_tokens__' % pref, x)]
            return f[c].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)

        def one(name):
            return pd.to_numeric(f[name], errors='coerce').fillna(0) if name in f.columns else pd.Series(0.0, index=f.index)
        r1t, r2t, r3t = (tk('r1'), tk('r2'), tk('r3'))
        o1, o2 = (one('organizer_after_r1_total_tokens'), one('organizer_after_r2_total_tokens'))
        syn = one('final_synthesis_total_tokens')
        pol = {'стоп@R1': (r1.loc[ix, 'Judge_score'].astype(float), r1t + o1 + syn), 'стоп@R2': (r2.loc[ix, 'Judge_score'].astype(float) if r2 is not None else None, r1t + r2t + o1 + o2 + syn), 'как есть': (f['Judge_score'].astype(float), r1t + r2t + r3t + o1 + o2 + syn)}
        base = pol['как есть']
        for name, (sc, tok) in pol.items():
            if sc is None:
                continue
            d = sc - base[0]
            p = stats.wilcoxon(d)[1] if (d != 0).any() else np.nan
            rows.append({'плечо': arm, 'политика': name, 'n': len(ix), 'Judge': sc.mean(), 'токенов/кейс': tok.mean(), 'Δ качества': d.mean(), 'p': p, 'экономия токенов': 1 - tok.mean() / base[1].mean(), 'Judge на 10k токенов': 10000 * sc.mean() / tok.mean()})
    t = pd.DataFrame(rows)
    print('\n[1] ПО ПЛЕЧАМ')
    print(t.set_index(['плечо', 'политика']).to_string())
    print('\n[2] СВОДКА ПО ПОЛИТИКАМ (среднее по 6 плечам)')
    s = t.groupby('политика').agg(Judge=('Judge', 'mean'), токенов=('токенов/кейс', 'mean'), Δкачества=('Δ качества', 'mean'), экономия=('экономия токенов', 'mean'), Judge_на_10k=('Judge на 10k токенов', 'mean'))
    s = s.reindex(['стоп@R1', 'стоп@R2', 'как есть'])
    print(s.to_string())
    print('\n[3] ОБЪЕДИНЁННЫЙ ТЕСТ «стоп@R1 против как есть» ПО ВСЕМ ПЛЕЧАМ')
    allд = []
    for arm, pat in ROUNDS.items():
        fin, r1 = (RAW.get(arm), rd(pat % 1))
        if fin is None or r1 is None:
            continue
        ix = fin.index.intersection(r1.index)
        allд.append(r1.loc[ix, 'Judge_score'].astype(float) - fin.loc[ix, 'Judge_score'].astype(float))
    a = pd.concat(allд)
    se = a.std(ddof=1) / np.sqrt(len(a))
    tost = max(stats.t.sf((a.mean() + 0.02) / se, len(a) - 1), stats.t.cdf((a.mean() - 0.02) / se, len(a) - 1))
    print('  n = %d, Δ = %+.4f, Уилкоксон p = %.3g, TOST ±0.02 p = %.3g' % (len(a), a.mean(), stats.wilcoxon(a)[1], tost))
    print('\n[4] СРАВНЕНИЕ С ОДИНОЧНЫМИ МОДЕЛЯМИ ПО ЦЕНЕ')
    br = []
    for m in ['gpt', 'qwen', 'deepseekv4']:
        d = RAW['base_' + m]
        c = [x for x in d.columns if x.endswith('total_tokens') and 'judge' not in x and ('uni' not in x)]
        tok = d[c].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1).mean()
        br.append({'плечо': 'baseline ' + m, 'Judge': d['Judge_score'].astype(float).mean(), 'токенов/кейс': tok, 'Judge на 10k токенов': 10000 * d['Judge_score'].astype(float).mean() / tok})
    u = RAW['USC@7']
    tu = pd.to_numeric(u['total_tokens'], errors='coerce').mean()
    br.append({'плечо': 'USC@7 gpt', 'Judge': u['Judge_score'].astype(float).mean(), 'токенов/кейс': tu, 'Judge на 10k токенов': 10000 * u['Judge_score'].astype(float).mean() / tu})
    for _, r in t[t['плечо'] == 'P-base'].iterrows():
        br.append({'плечо': 'P-base, ' + r['политика'], 'Judge': r['Judge'], 'токенов/кейс': r['токенов/кейс'], 'Judge на 10k токенов': r['Judge на 10k токенов']})
    print(pd.DataFrame(br).set_index('плечо').to_string())
    t.to_csv(os.path.normpath(os.path.join(OUT, 'early_stopping_policy_costs.csv')), index=False, encoding='utf-8-sig')
    return t

def cmh(tables):
    tables = [t for t in tables if t.sum() > 0]
    num = sum((t[0, 0] * t[1, 1] / t.sum() for t in tables))
    den = sum((t[0, 1] * t[1, 0] / t.sum() for t in tables))
    if den == 0 or num == 0:
        return (np.nan, np.nan)
    orr = num / den
    S = sum((t[0].sum() * t[:, 0].sum() / t.sum() for t in tables))
    O = sum((t[0, 0] for t in tables))
    V = sum((t[0].sum() * t[1].sum() * t[:, 0].sum() * t[:, 1].sum() / (t.sum() ** 2 * (t.sum() - 1)) for t in tables if t.sum() > 1))
    chi = (abs(O - S) - 0.5) ** 2 / V if V > 0 else np.nan
    return (orr, stats.chi2.sf(chi, 1) if np.isfinite(chi) else np.nan)

def h37():
    print('\n' + '=' * 108)
    print('H3.7  РАВНОМЕРЕН ЛИ ПРИРОСТ ПАНЕЛИ ПО ТРУДНОСТИ ПУНКТА')
    print('=' * 108)
    print('ПОПРАВКА К ПЛАНУ: 2PL-IRT здесь неидентифицируема (2334 пункта против 14 плеч).')
    print('Вместо неё: трудность оценена на ОТЛОЖЕННЫХ плечах, контраст стратифицирован')
    print('по пункту (Мантель-Хензель), однородность OR по стратам -- хи-квадрат.')
    mem = PANEL_MEMBERS['P-base']
    held = [m for m in SINGLES if m not in mem]
    rows = []
    for k in POS:
        rows.append({'case': k[0], 'cid': k[1], 'diff': 1 - np.mean([ITEMS['base_' + m][k][0] for m in held]), 'panel': ITEMS['P-base'][k][0], 'mem_mean': np.mean([ITEMS['base_' + m][k][0] for m in mem]), 'best_mem': max((ITEMS['base_' + m][k][0] for m in mem)), 'points': ITEMS['P-base'][k][1]})
    d = pd.DataFrame(rows)
    d['strat'] = pd.cut(d['diff'], [-0.01, 0.001, 0.26, 0.51, 0.76, 1.01], labels=['очень лёгкие', 'лёгкие', 'средние', 'трудные', 'очень трудные'])
    print('\n[1] ДОЛЯ ВЫПОЛНЕННЫХ ПО СТРАТАМ ТРУДНОСТИ (трудность -- по 4 отложенным плечам)')
    t = d.groupby('strat', observed=True).agg(пунктов=('panel', 'size'), панель=('panel', 'mean'), участники_среднее=('mem_mean', 'mean'), лучший_участник=('best_mem', 'mean'))
    t['панель − среднее'] = t['панель'] - t['участники_среднее']
    t['панель − лучший'] = t['панель'] - t['лучший_участник']
    print(t.to_string())
    print('\n[2] МАНТЕЛЬ-ХЕНЗЕЛЬ: панель против ЛУЧШЕГО участника, стратификация по пункту')
    res = []
    for lab, s in d.groupby('strat', observed=True):
        tabs = []
        for _, r in s.iterrows():
            tabs.append(np.array([[int(r.panel), 1 - int(r.panel)], [int(r.best_mem), 1 - int(r.best_mem)]]))
        o, p = cmh(tabs)
        b = int((s.panel & ~s.best_mem.astype(bool)).sum())
        c = int((~s.panel.astype(bool) & s.best_mem.astype(bool)).sum())
        pm = stats.binomtest(b, b + c, 0.5).pvalue if b + c else np.nan
        res.append({'страта': lab, 'пунктов': len(s), 'OR (МХ)': o, 'панель+/лучший−': b, 'панель−/лучший+': c, 'p (McNemar)': pm})
    r7 = pd.DataFrame(res).set_index('страта')
    print(r7.to_string())
    print('\n[3] ОДНОРОДЕН ЛИ ЭФФЕКТ ПО СТРАТАМ')
    ct = pd.crosstab(d.strat, [d.panel, d.best_mem])
    disc = d[d.panel != d.best_mem]
    ct2 = pd.crosstab(disc.strat, disc.panel)
    chi = stats.chi2_contingency(ct2)
    print('  доля «панель выиграла» среди расходящихся пунктов, по стратам:')
    print('  ' + (ct2[True] / ct2.sum(axis=1)).round(3).to_string().replace('\n', '\n  '))
    print('  хи-квадрат на однородность: chi2 = %.2f, df = %d, p = %.3g' % (chi.statistic, chi.dof, chi.pvalue))
    print('\n[4] ЛОГИСТИЧЕСКАЯ МОДЕЛЬ С ВЗАИМОДЕЙСТВИЕМ (пункт -- единица наблюдения)')
    long = pd.concat([pd.DataFrame({'met': d.panel.astype(int), 'arm': 'panel', 'diff': d['diff'], 'points': d.points, 'case': d.case}), pd.DataFrame({'met': d.best_mem.astype(int), 'arm': 'best_mem', 'diff': d['diff'], 'points': d.points, 'case': d.case})])
    m = smf.logit("met ~ C(arm, Treatment('best_mem')) * diff + points", long).fit(disp=0)
    tab = pd.DataFrame({'коэф': m.params, 'SE': m.bse, 'z': m.tvalues, 'p': m.pvalues, 'OR': np.exp(m.params)})
    print(tab.to_string())
    print('  n = %d наблюдений (%d пунктов x 2 плеча), псевдо-R2 = %.3f' % (int(m.nobs), len(d), m.prsquared))
    print('\n[5] ТО ЖЕ ДЛЯ ВСЕХ ПАНЕЛЕЙ: OR панель/лучший участник по стратам трудности')
    out = []
    for arm, mm in PANEL_MEMBERS.items():
        if arm not in ITEMS:
            continue
        hd = [m2 for m2 in SINGLES if m2 not in mm]
        rr = []
        for k in POS:
            rr.append({'diff': 1 - np.mean([ITEMS['base_' + m2][k][0] for m2 in hd]), 'panel': ITEMS[arm][k][0], 'best_mem': max((ITEMS['base_' + m2][k][0] for m2 in mm))})
        dd = pd.DataFrame(rr)
        dd['strat'] = pd.cut(dd['diff'], [-0.01, 0.001, 0.51, 1.01], labels=['лёгкие', 'средние', 'трудные'])
        row = {'панель': arm}
        for lab, s in dd.groupby('strat', observed=True):
            b = int((s.panel & ~s.best_mem.astype(bool)).sum())
            c = int((~s.panel.astype(bool) & s.best_mem.astype(bool)).sum())
            row[str(lab)] = '%d/%d OR=%.2f' % (b, c, b / c if c else np.inf)
        out.append(row)
    print(pd.DataFrame(out).set_index('панель').to_string())
    print('  формат: выиграла/проиграла, OR -- отношение шансов панели к лучшему участнику')
    d.to_csv(os.path.normpath(os.path.join(OUT, 'criterion_difficulty_and_panel_gain.csv')), index=False, encoding='utf-8-sig')
    return d
if __name__ == '__main__':
    h32()
    h33()
    h37()
    print('\nCSV: %s' % OUT)


ЗАГРУЗКА ПОПУНКТНЫХ ВЕРДИКТОВ


плеч загружено: 14  (P-base, P-weak, P-mixed, P-strong, homog_gpt, homog_ds, USC@7, base_gpt, base_qwen, base_deepseekv4, base_deepseekv4pro, base_gemini, base_glm, base_kimi)
сопоставлено положительных пунктов во ВСЕХ 14 плечах: 2249

H3.2  ЧТО ИМЕННО ПОРОЖДАЕТ ПАНЕЛЬ
Подвыборка: пункты, которые НЕ выполнил ни один из трёх участников панели. n = 623, восстановлено 246 (39.5%).
Внешняя трудность ext_pass -- доля из 4 плеч ВНЕ панели (deepseekv4pro, gemini, glm, kimi), выполнивших пункт. Циркулярности нет: эти плечи в панель не входят.

[1] ВОССТАНОВЛЕНИЕ ПО ВНЕШНЕЙ ТРУДНОСТИ
                      пунктов  восстановлено  средний_вес  восстановлено_шт
ext_bin                                                                    
0/4 никто вне панели      482       0.354772     5.371369               171
1/4                       112       0.473214     5.866071                53
2/4                        24       0.708333     6.250000                17
3/4                         4       1.

                       пунктов  восстановлено  внешняя_трудность
axis                                                            
accuracy                    58       0.362069           0.099138
communication_quality       68       0.264706           0.077206
completeness               281       0.441281           0.075623
reasoning                   28       0.571429           0.169643
uncertainty                161       0.341615           0.020186
urgency                     27       0.444444           0.138889

[5] ЛЕКСИКА: чем восстановленные пункты отличаются от невосстановленных


  чаще у ВОССТАНОВЛЕННЫХ: mentions, given, disease, abscess, age, differential, appropriate, blood, workup, if, evaluation, as
  чаще у ПРОПУЩЕННЫХ:     asks, asks about, about, prior, and, to, family, history, use, that, about prior, recent

H3.3  ЦЕНА ОСТАНОВКИ ПОСЛЕ R1 КАК ПОЛИТИКА
Политика «как есть» = ранняя остановка по единогласию (то, что прогонялось).
Токены политики «стоп@Rk» = все вызовы экспертов по раунды 1..k + сводки
организатора после этих раундов + один финальный синтез.



[1] ПО ПЛЕЧАМ
                      n     Judge  токенов/кейс  Δ качества         p  экономия токенов  Judge на 10k токенов
плечо     политика                                                                                           
P-base    стоп@R1   229  0.783719  18418.161572   -0.006227  0.400175          0.311379              0.425514
          стоп@R2   229  0.797462  24521.650655    0.007515  0.027330          0.083181              0.325207
          как есть  229  0.789946  26746.454148    0.000000       NaN          0.000000              0.295346
P-weak    стоп@R1   229  0.747312  16066.353712    0.002005  0.834773          0.341552              0.465141
          стоп@R2   229  0.743877  21879.868996   -0.001431  0.602605          0.103296              0.339982
          как есть  229  0.745307  24400.327511    0.000000       NaN          0.000000              0.305450
P-mixed   стоп@R1   229  0.793006  17746.240175    0.006603  0.221159          0.220221              0.44

  n = 1374, Δ = +0.0004, Уилкоксон p = 0.79, TOST ±0.02 p = 2.59e-21

[4] СРАВНЕНИЕ С ОДИНОЧНЫМИ МОДЕЛЯМИ ПО ЦЕНЕ
                        Judge  токенов/кейс  Judge на 10k токенов
плечо                                                            
baseline gpt         0.625811   1752.008734              3.571962
baseline qwen        0.562228   1250.790393              4.494981
baseline deepseekv4  0.640056   1811.771930              3.532761
USC@7 gpt            0.627953  27521.602620              0.228167
P-base, стоп@R1      0.783719  18418.161572              0.425514
P-base, стоп@R2      0.797462  24521.650655              0.325207
P-base, как есть     0.789946  26746.454148              0.295346

H3.7  РАВНОМЕРЕН ЛИ ПРИРОСТ ПАНЕЛИ ПО ТРУДНОСТИ ПУНКТА
ПОПРАВКА К ПЛАНУ: 2PL-IRT здесь неидентифицируема (2334 пункта против 14 плеч).
Вместо неё: трудность оценена на ОТЛОЖЕННЫХ плечах, контраст стратифицирован
по пункту (Мантель-Хензель), однородность OR по стратам -- хи-квадрат.



[1] ДОЛЯ ВЫПОЛНЕННЫХ ПО СТРАТАМ ТРУДНОСТИ (трудность -- по 4 отложенным плечам)
               пунктов    панель  участники_среднее  лучший_участник  панель − среднее  панель − лучший
strat                                                                                                  
очень лёгкие       727  0.987620           0.962861         0.998624          0.024759        -0.011004
лёгкие             267  0.955056           0.822722         0.985019          0.132335        -0.029963
средние            234  0.820513           0.618234         0.897436          0.202279        -0.076923
трудные            339  0.681416           0.369715         0.669617          0.311701         0.011799
очень трудные      682  0.435484           0.128055         0.293255          0.307429         0.142229

[2] МАНТЕЛЬ-ХЕНЗЕЛЬ: панель против ЛУЧШЕГО участника, стратификация по пункту


               пунктов   OR (МХ)  панель+/лучший−  панель−/лучший+   p (McNemar)
страта                                                                          
очень лёгкие       727  0.111111                1                9  2.148438e-02
лёгкие             267  0.333333                4               12  7.681274e-02
средние            234  0.485714               17               35  1.753324e-02
трудные            339  1.081633               53               49  7.666043e-01
очень трудные      682  2.310811              171               74  5.158666e-10

[3] ОДНОРОДЕН ЛИ ЭФФЕКТ ПО СТРАТАМ
  доля «панель выиграла» среди расходящихся пунктов, по стратам:
  strat
  очень лёгкие     0.100
  лёгкие           0.250
  средние          0.327
  трудные          0.520
  очень трудные    0.698
  хи-квадрат на однородность: chi2 = 45.77, df = 4, p = 2.75e-09

[4] ЛОГИСТИЧЕСКАЯ МОДЕЛЬ С ВЗАИМОДЕЙСТВИЕМ (пункт -- единица наблюдения)


                                                 коэф        SE          z             p         OR
Intercept                                    4.533172  0.343156  13.210224  7.659566e-40  93.053284
C(arm, Treatment('best_mem'))[T.panel]      -1.628807  0.337930  -4.819962  1.435855e-06   0.196163
diff                                        -6.267091  0.322413 -19.438059  3.678228e-84   0.001898
C(arm, Treatment('best_mem'))[T.panel]:diff  2.241685  0.386764   5.796000  6.791536e-09   9.409171
points                                       0.149861  0.028527   5.253317  1.493838e-07   1.161673
  n = 4498 наблюдений (2249 пунктов x 2 плеча), псевдо-R2 = 0.378

[5] ТО ЖЕ ДЛЯ ВСЕХ ПАНЕЛЕЙ: OR панель/лучший участник по стратам трудности


                 лёгкие        средние          трудные
панель                                                 
P-base      1/9 OR=0.11  21/47 OR=0.45  224/123 OR=1.82
P-weak     8/14 OR=0.57  87/50 OR=1.74   224/79 OR=2.84
P-mixed     1/7 OR=0.14  60/42 OR=1.43   250/89 OR=2.81
P-strong    0/6 OR=0.00  21/33 OR=0.64  244/127 OR=1.92
homog_gpt  13/8 OR=1.62  96/34 OR=2.82   319/63 OR=5.06
homog_ds   13/2 OR=6.50  80/27 OR=2.96   335/60 OR=5.58
  формат: выиграла/проиграла, OR -- отношение шансов панели к лучшему участнику

CSV: .\analyses\panel_mechanisms\criterion_recovery_and_difficulty
